In [1]:
#Importing libraries

import glob
from Bio import SeqIO
from collections import Counter
import pandas as pd 
import os
from pathlib import Path

In [ ]:
#Set the path of directory containing genbank files
file_directory=Path(r'C:\Users\SB\Documents\GENEBANK')

In [3]:
#Get the file names
gfiles=glob.glob('%s/*.gb'%file_directory)
print(gfiles)

['C:\\Users\\SB\\Documents\\GENEBANK\\1.gb', 'C:\\Users\\SB\\Documents\\GENEBANK\\sequence (2).gb', 'C:\\Users\\SB\\Documents\\GENEBANK\\sequence (3).gb', 'C:\\Users\\SB\\Documents\\GENEBANK\\sequence (4).gb', 'C:\\Users\\SB\\Documents\\GENEBANK\\sequence.gb']


In [4]:
gfiles[0]

'C:\\Users\\SB\\Documents\\GENEBANK\\1.gb'

In [ ]:
#function to read genbank files and get the features
# Import SeqIO from BioPython to read molecular biology file formats

def read_file(gfile):
    genbank_object=SeqIO.read(gfile,'gb')                     # Parse the GenBank file using the 'gb' format specifier
    features=genbank_object.features                          # Extract the features list from the parsed GenBank object
    feature_types=[feature.type for feature in features]      # Use list comprehension to get the string type name of each feature
    return feature_types

In [ ]:
#count features of the genbank files
# Import Counter to easily tally occurrences of items in a list
def count_features(feature_types):                    #Counts the frequency of each feature type within a file
    feature_count=Counter(feature_types)              # Create a frequency dictionary-like object mapping feature names to counts
    print('features have been counted')
    return feature_count

In [ ]:
#function to identify features in genbank files

def scan_all_features(files):                          #Scans all input files to generate a master list of unique features
    allfeatures=[]
    for gfile in gfiles:                                # Loop through each GenBank file to gather all present feature types
        feature_types=read_file(gfile)
        allfeatures.extend(feature_types)

    allfeatures=set(allfeatures)
    allfeatures=list(allfeatures)
    print('all features have been identified')
    return allfeatures

In [8]:
#get all features

allfeatures=scan_all_features(gfiles)
print(allfeatures)


all features have been identified
['ncRNA', 'tRNA', 'misc_binding', 'rRNA', 'CDS', 'source', 'tmRNA', 'gene', 'regulatory']


In [9]:
#create an empty list this will be used to store the counts
allfeature_count=[]

In [10]:
#use for loop to iterate over the files and count the features
for gfile in gfiles:
    directory,filename=os.path.split(gfile)
    filename=filename.strip('.gb')
    feature_types=read_file(gfile)
    feature_count=count_features(feature_types)
    temp_count=[]

    temp_count.append(filename)

    for feature in allfeatures:
        if feature in feature_count.keys():
            temp_count.append(feature_count[feature])
        else:
            temp_count.append(0)

    allfeature_count.append(temp_count)

features have been counted
features have been counted
features have been counted
features have been counted
features have been counted


In [11]:
len(allfeature_count)
allfeature_count[0]

['1', 3, 60, 0, 16, 3171, 1, 1, 3251, 10]

In [12]:
print(allfeatures)

['ncRNA', 'tRNA', 'misc_binding', 'rRNA', 'CDS', 'source', 'tmRNA', 'gene', 'regulatory']


In [13]:
allfeature_count

[['1', 3, 60, 0, 16, 3171, 1, 1, 3251, 10],
 ['sequence (2)', 0, 60, 0, 19, 2740, 1, 0, 2819, 0],
 ['sequence (3)', 3, 59, 0, 10, 2935, 1, 1, 3008, 10],
 ['sequence (4)', 3, 60, 10, 16, 3114, 1, 1, 3194, 0],
 ['sequence', 3, 59, 0, 8, 3179, 1, 1, 3250, 10]]

In [14]:
#Create columns for the dataframe
columns=[]
columns.append('File')
columns.extend(allfeatures)
columns

['File',
 'ncRNA',
 'tRNA',
 'misc_binding',
 'rRNA',
 'CDS',
 'source',
 'tmRNA',
 'gene',
 'regulatory']

In [ ]:
#Create dataframe using pandas

dataframe=pd.DataFrame(allfeature_count,columns=columns)
print(dataframe)

           File  ncRNA  tRNA  misc_binding  rRNA   CDS  source  tmRNA  gene  \
0             1      3    60             0    16  3171       1      1  3251   
1  sequence (2)      0    60             0    19  2740       1      0  2819   
2  sequence (3)      3    59             0    10  2935       1      1  3008   
3  sequence (4)      3    60            10    16  3114       1      1  3194   
4      sequence      3    59             0     8  3179       1      1  3250   

   regulatory  
0          10  
1           0  
2          10  
3           0  
4          10  


In [16]:
#Set index to 'File'

finaldata=dataframe.set_index('File')
finaldata

,ncRNA,tRNA,misc_binding,rRNA,CDS,source,tmRNA,gene,regulatory
File,,,,,,,,,
1,3,60,0,16,3171,1,1,3251,10
sequence (2),0,60,0,19,2740,1,0,2819,0
sequence (3),3,59,0,10,2935,1,1,3008,10
sequence (4),3,60,10,16,3114,1,1,3194,0
sequence,3,59,0,8,3179,1,1,3250,10


In [18]:
columns_to_delete=['source','misc_binding','regulatory']

In [19]:
#Remove columnsfrom dataframe

finaldata.drop(columns=columns_to_delete,inplace=True)
print(finaldata)

              ncRNA  tRNA  rRNA   CDS  tmRNA  gene
File                                              
1                 3    60    16  3171      1  3251
sequence (2)      0    60    19  2740      0  2819
sequence (3)      3    59    10  2935      1  3008
sequence (4)      3    60    16  3114      1  3194
sequence          3    59     8  3179      1  3250


In [20]:
finaldata.loc['1',:]


ncRNA       3
tRNA       60
rRNA       16
CDS      3171
tmRNA       1
gene     3251
Name: 1, dtype: int64

In [21]:
finaldata.loc['1','gene']

np.int64(3251)

In [27]:
#save finaldata to an output file
#define the output filename
outputfile=r'C:\Users\SB\Documents\GENEBANK\featurecount.csv'

In [28]:
#Save finaldata to a file(csv format)
finaldata.to_csv(outputfile,index=True)